# accumap on the experimental ground truth (Fig. 2, Supp Figs S2-S3, Supp Tables S3-S4)

The four replicates of Hu et al. (2025) are the only optics-free arrays with imaging ground truth. This notebook reconstructs them with accumap and builds **Fig. 2** and **Supp Table S4**, followed by **Supp Figs S2-S3**, which explain why the method needs physical priors.

**Runtime.** By default (`RECOMPUTE = False`) the notebook draws everything from the cached results in `paper_results/` in about a minute. If the Hu data and a CUDA GPU are available, it also runs one reconstruction live (replicate c58_8, UMAP start 0; about 1 minute) to show the method end to end. `RECOMPUTE = True` reruns all 32 reconstructions (4 replicates x 8 UMAP starts, about 1 hour on one GPU) and recomputes every number.

**Data.** For the live run or recomputation, download `<rep>_blind_raw_reads_filtered.csv.gz` and `<rep>_matched_bead_location.csv.gz` for each replicate from the Single Cell Portal study [SCP2577](https://singlecell.broadinstitute.org/single_cell/study/SCP2577/computational-reconstruction-for-imaging-free-spatial-transcriptomics) into the repository's `data/` folder (or set `ACCUMAP_DATA` to their directory). Install the package with the count-model extra: `pip install -e "..[countmodel]"`.

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from accumap.metrics import evaluate, neighbors_kept, similarity_align

RECOMPUTE = False                      # True: rerun all 32 reconstructions (GPU, about 1 h)
RESULTS = "paper_results"              # cached results shipped with the repository
FIGURES = "../final_figures"
os.makedirs(FIGURES, exist_ok=True)
REPS = ("c58_8", "c58_4", "c58_5", "c58_16")
SEEDS = range(8)

try:
    import torch
    GPU = torch.cuda.is_available()
except ImportError:
    GPU = False
from accumap.real_data import DATA_DIR
have = lambda r: os.path.exists(os.path.join(DATA_DIR, f"{r}_matched_bead_location.csv.gz"))
HAVE_C58_8, HAVE_ALL = have("c58_8"), all(have(r) for r in REPS)
print(f"GPU: {GPU}, c58_8 data: {HAVE_C58_8}, all replicates: {HAVE_ALL}, RECOMPUTE: {RECOMPUTE}")
assert not RECOMPUTE or (GPU and HAVE_ALL), "RECOMPUTE needs a CUDA GPU and all four replicates in " + DATA_DIR

def umap_start(rep, seed):
    """The cached UMAP initialization (receivers X0, senders U0) used for every run in the paper."""
    f = "c58_8_joint_proto_umap.npz" if (rep, seed) == ("c58_8", 0) else f"{rep}_umapinit_seed{seed}.npz"
    return np.load(os.path.join(RESULTS, "umap_starts", f))["X0"]

## One reconstruction, end to end

`reconstruct` takes the count matrix, a starting layout (UMAP) and the array's physical diameter, and nothing else: no imaging is used. The ground truth is used only afterwards, to score the result.

In [ ]:
from accumap.real_data import load_hu

if GPU and HAVE_C58_8:
    from accumap.count_model import reconstruct
    C, X_true, _ = load_hu("c58_8", count="umi")
    E = umap_start("c58_8", 0)
    R = reconstruct(C, E, array_diameter=3000.0, verbose=True)
    print("UMAP   ", {k: round(v, 3) for k, v in evaluate(E, X_true).items()})
    print("accumap", {k: round(v, 3) for k, v in evaluate(R["receivers"], X_true).items()})
else:
    print("Skipping the live run (needs the c58_8 data and a CUDA GPU); using cached results below.")

## All replicates and UMAP starts

Each replicate is reconstructed from 8 UMAP initializations. With `RECOMPUTE = False` the cached per-run metrics are loaded; otherwise all 32 runs are redone and the cache is overwritten.

In [ ]:
def fmt(m):   # metrics dict -> the compact string stored in the cache
    return f"recv p50 {m['median']:.1f} p90 {m['p90']:.1f} pairRMSE {m['pairwise_rmse']:.1f} J@10 {m['jaccard10']:.3f}"

if RECOMPUTE:
    from accumap.count_model import reconstruct
    rows, extras = [], {}
    for rep in REPS:
        C, X_true, _ = load_hu(rep, count="umi")
        kept_u, kept_a = [], []
        for s in SEEDS:
            E = umap_start(rep, s)
            X = reconstruct(C, E, array_diameter=3000.0, verbose=False)["receivers"]
            rows.append({"rep": rep, "seed": s, "umap": fmt(evaluate(E, X_true)), "v2": fmt(evaluate(X, X_true))})
            kept_u.append(neighbors_kept(E, X_true)); kept_a.append(neighbors_kept(X, X_true))
            if s == 0:
                extras[rep] = (X_true, E, X)
                if rep == "c58_8":
                    np.savez_compressed(os.path.join(RESULTS, "c58_8_v2_seed0.npz"), X=X)
            print(rep, s, rows[-1]["v2"], flush=True)
        extras[rep + "_kept"] = (kept_u, kept_a)
    with open(os.path.join(RESULTS, "hu_v2.jsonl"), "w") as fh:
        fh.writelines(json.dumps(r) + "\n" for r in rows)
    nn10 = {}
    for rep in REPS:
        ku, ka = extras[rep + "_kept"]
        pu = [100 * (k >= 9).mean() for k in ku]; pa = [100 * (k >= 9).mean() for k in ka]
        nn10[rep] = {"umap_pct9": float(np.mean(pu)), "acc_pct9": float(np.mean(pa)), "umap_mean": float(np.mean([k.mean() for k in ku])),
                     "acc_mean": float(np.mean([k.mean() for k in ka])), "better": int(sum(x > y for x, y in zip(pa, pu)))}
    json.dump(nn10, open(os.path.join(RESULTS, "nn10_table.json"), "w"), indent=1)

parse = lambda m: [float(x) for x in re.findall(r"(?:p50|p90|pairRMSE|J@10) ([\d.]+)", m)]
runs = [json.loads(l) for l in open(os.path.join(RESULTS, "hu_v2.jsonl"))]
nn10 = json.load(open(os.path.join(RESULTS, "nn10_table.json")))
print(len(runs), "runs loaded")

## Fig. 2

Data for the figure: the c58_8 truth and reconstructions from UMAP start 0 (panels A-C, E), and per-bead errors on all four replicates (panel D). These come from the cache, or from the recomputation above.

In [ ]:
if RECOMPUTE:
    d = {"c58_8_truth": extras["c58_8"][0]}
    for rep in REPS:
        X_true, E, X = extras[rep]
        for key, P in (("umap", E), ("method", X)):
            A, T = similarity_align(P, X_true)
            d[f"{rep}_err_{key}"] = np.linalg.norm(A - T, axis=1)
else:
    d = dict(np.load(os.path.join(RESULTS, "mech_fig_data.npz")))
T = d["c58_8_truth"]
recon = {"accumap": np.load(os.path.join(RESULTS, "c58_8_v2_seed0.npz"))["X"], "UMAP": umap_start("c58_8", 0)}

TITLE_FS, LABEL_FS, TICK_FS, LETTER_FS = 13, 12, 10, 18
C_UMAP, C_METHOD, C_REF = "#dc143c", "#4169e1", "#8a8986"
CHECKER = ("#1a1a1a", "#eb6834")          # near-black / orange
SQUARE = 250.0                             # checker square side, um
MU = "$\\mu$m"

def align(P, T):
    Pc, Tc = P - P.mean(0), T - T.mean(0)
    U, s, Vt = np.linalg.svd(Pc.T @ Tc)
    return Pc @ (U @ Vt) * (s.sum() / np.sum(Pc ** 2)), Tc


def style(ax):
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=TICK_FS)


def letter(ax, s, dx=-0.18, dy=1.04):
    ax.text(dx, dy, s, transform=ax.transAxes, fontsize=LETTER_FS, fontweight="bold", va="bottom")

In [ ]:
fig = plt.figure(figsize=(17.5, 10.5))
gs = fig.add_gridspec(2, 1, height_ratios=[1.25, 1.0], hspace=0.22)
top = gs[0].subgridspec(1, 3, wspace=0.05)
bot = gs[1].subgridspec(1, 3, width_ratios=[4.0, 0.42, 1.9])
cdfs = bot[0].subgridspec(1, 4, wspace=0.12)

# A-C: each receiver colored by whether it keeps >= 9 of its 10 true nearest neighbors (teal) or not (black)
from scipy.spatial import cKDTree
from matplotlib.patches import Patch
GREEN, BLACK = "#5fd8a4", "#1a1a1a"
Tc = T - T.mean(0)
true_nn = cKDTree(Tc).query(Tc, 11)[1][:, 1:]
panels, kept = [("ground truth", Tc, None)], {"ground truth": np.full(len(Tc), 10)}
for name in ("accumap", "UMAP"):
    A, _ = align(recon[name], T)
    panels.append((name, A, np.median(np.linalg.norm(A - Tc, axis=1))))
    rec_nn = cKDTree(A).query(A, 11)[1][:, 1:]
    kept[name] = np.array([len(set(x) & set(y)) for x, y in zip(true_nn, rec_nn)])
order = np.random.default_rng(0).permutation(len(Tc))      # random drawing order: neither color hides the other
lim = np.abs(Tc).max() * 1.08
for k, (name, P, med) in enumerate(panels):
    ax = fig.add_subplot(top[k])
    col = np.where(kept[name] >= 9, GREEN, BLACK)
    ax.scatter(P[order, 0], P[order, 1], c=col[order], s=1.8, linewidths=0, rasterized=True)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_title(name if med is None else f"{name}\nmedian error {med:.1f} {MU}", fontsize=TITLE_FS)
    letter(ax, "ABC"[k], dx=0.0, dy=1.0)
    if k == 0:
        ax.plot([-lim * 0.92, -lim * 0.92 + 500], [-lim * 0.97] * 2, color="k", lw=2)
        ax.text(-lim * 0.92 + 250, -lim * 0.99, f"500 {MU}", ha="center", va="top", fontsize=TICK_FS)
        ax.legend(handles=[Patch(facecolor=GREEN, label="\u2265 9 of 10 true\nneighbors kept"), Patch(facecolor=BLACK, label="fewer")],
                  fontsize=TICK_FS, frameon=False, loc="upper left", bbox_to_anchor=(-0.22, 0.97))

# D: per-bead error CDFs
for k, rep in enumerate(REPS):
    ax = fig.add_subplot(cdfs[k])
    for e, col, lab in ((d[f"{rep}_err_umap"], C_UMAP, "UMAP"), (d[f"{rep}_err_method"], C_METHOD, "accumap")):
        e = np.sort(e)
        ax.plot(e, np.arange(1, len(e) + 1) / len(e), color=col, lw=2, label=lab)
    ax.set_xlim(0, 150); ax.set_ylim(0, 1.0); ax.set_xticks([0, 50, 100])
    ax.set_title(rep, fontsize=TITLE_FS)
    cdf_axes = [ax] if k == 0 else cdf_axes + [ax]
    if k == 0:
        ax.set_ylabel("Fraction of beads", fontsize=LABEL_FS)
        letter(ax, "D", dx=-0.42)
        ax.legend(fontsize=TICK_FS, frameon=False, loc="lower right")
    else:
        ax.set_yticklabels([])
    style(ax)

# E: true neighbors kept (of 10), c58_8, this UMAP initialization; >= 9 kept in teal
ax = fig.add_subplot(bot[2])
v = np.arange(11); w = 0.4
colors = [GREEN if x >= 9 else BLACK for x in v]
for j_, (name, hatch) in enumerate((("accumap", None), ("UMAP", "///"))):
    frac = np.bincount(kept[name], minlength=11) / len(kept[name])
    edges = ["white" if (hatch and c_ == BLACK) else BLACK for c_ in colors]
    ax.bar(v + (j_ - 0.5) * w, 100 * frac, width=w, color=colors, edgecolor=edges, linewidth=0.6, hatch=hatch)
ax.set_xticks(range(0, 11, 2)); ax.set_xlim(2.4, 10.6)
ax.set_xlabel("true neighbors kept (of 10)", fontsize=LABEL_FS)
ax.set_ylabel("% of receivers (c58_8)", fontsize=LABEL_FS)
ax.legend(handles=[Patch(facecolor="white", edgecolor=BLACK, label="accumap"),
                   Patch(facecolor="white", edgecolor=BLACK, hatch="///", label="UMAP"),
                   Patch(facecolor=GREEN, edgecolor=BLACK, label="\u2265 9 kept")],
          fontsize=TICK_FS, frameon=False, loc="upper left")
style(ax); letter(ax, "E", dx=-0.32)

# one shared x-axis title for the four CDF panels
p0, p1 = cdf_axes[0].get_position(), cdf_axes[-1].get_position()
fig.text((p0.x0 + p1.x1) / 2, p0.y0 - 0.03, f"Per-bead position error ({MU})", ha="center", va="top", fontsize=LABEL_FS)
fig.savefig(os.path.join(FIGURES, "Fig2_final.png"), dpi=300, bbox_inches="tight")
plt.show()

## Supp Table S4

Mean and SD over the 8 UMAP starts per replicate; the paired difference is accumap minus UMAP, and "Improved" counts starts on which accumap was better.

In [ ]:
names = ["median (um)", "90th pct. (um)", "pairwise RMSE (um)", "10-NN Jaccard"]
rows = []
for rep in REPS:
    G = sorted([r for r in runs if r["rep"] == rep], key=lambda r: r["seed"])
    U = np.array([parse(r["umap"]) for r in G]); V = np.array([parse(r["v2"]) for r in G]); D = V - U
    for i, n in enumerate(names):
        rows.append({"replicate": rep, "metric": n, "UMAP": f"{U[:, i].mean():.3g} ± {U[:, i].std(ddof=1):.2g}",
                     "accumap": f"{V[:, i].mean():.3g} ± {V[:, i].std(ddof=1):.2g}",
                     "paired difference": f"{D[:, i].mean():+.3g} ± {D[:, i].std(ddof=1):.2g}",
                     "improved": f"{int((D[:, i] < 0).sum())}/{len(G)}"})
table_s4 = pd.DataFrame(rows).set_index(["replicate", "metric"])
table_s4

## Why the method needs physical priors (Supp Figs S2, S3)

These two figures explain how accumap is built rather than how well it performs. Every analysis behind them uses the imaging ground truth: fitting the kernel at the true positions, localizing receivers with the true sender positions, and decomposing UMAP's true displacement field. They are therefore **drawn from cached results only**; the analyses themselves are not part of accumap and are not rerun here. Constants in the next cell come from those diagnostic runs (documented in Supplementary Note 2).

In [ ]:
d = dict(np.load(os.path.join(RESULTS, 'mech_fig_data.npz')))
TITLE_FS, LABEL_FS, TICK_FS, LETTER_FS = 13, 12, 10, 18
C_UMAP, C_METHOD, C_ALT, C_ALT2 = "#dc143c", "#4169e1", "#1baf7a", "#eda100"
C_REF = "#8a8986"  # neutral reference lines / seed connectors
REPS = ("c58_8", "c58_4", "c58_5", "c58_16")
MU = "$\\mu$m"

# displacement covariance of aligned embeddings vs true separation, normalized by
# mean squared error (umap_error_structure.log); 0-25um bin omitted (see caption).
COV_BINS = [(25, 50), (50, 100), (100, 200), (200, 400), (400, 800), (800, 1600)]
COV_UMAP = [0.59, 0.57, 0.51, 0.41, 0.21, -0.11]
COV_IID = [-0.02, 0.02, 0.01, -0.00, -0.00, -0.00]
# UMAP + true smooth displacement component (warp_decomp.log): % of the UMAP->truth
# log-likelihood gain recovered, median receiver error.
DECOMP = [("UMAP", 0, 38.6), ("+ warp, 400 " + MU, 1, 13.5), ("200 " + MU, 1, 12.2),
          ("100 " + MU, -3, 11.1), ("50 " + MU, -1, 9.7), ("truth", 100, 0.0)]



def style(ax):
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=TICK_FS)


def letter(ax, s, dx=-0.18, dy=1.06):
    ax.text(dx, dy, s, transform=ax.transAxes, fontsize=LETTER_FS, fontweight="bold", va="bottom")


def cdf(ax, e, color, lw=2, **kw):
    e = np.sort(e)
    ax.plot(e, np.arange(1, len(e) + 1) / len(e), color=color, lw=lw, **kw)

### Supp Fig S2: the transfer kernel

(A) The kernel fit by Poisson maximum likelihood at the imaging positions, and the imaging-free kernel accumap fits on the UMAP layout (Fig. 1D). Transfer is flat to about 20 µm, falls twelve-fold by 35 µm, then decays as a heavy tail; the imaging-free kernel is broader because UMAP's errors blur it. (B) Cumulative fraction of UMIs by the distance between receiver and sender: transfer is dominated by contact between neighboring beads.

In [ ]:
# ============================================ Supp Fig S2: transfer kernel ==
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.6))
knots = d["knots"]
ax = axes[0]
for key, col, lab in (("logK_imaging_poisson", C_METHOD, "fit to imaging"),
                      ("logK_selffit", C_ALT, "imaging-free (fit to UMAP layout)")):
    m = knots <= 1500  # outermost knot (4000um) is beyond the fitted data: extrapolation, not shown
    ax.plot(knots[m], np.exp(d[key][m]), "o-", color=col, lw=2, ms=5, label=lab)
K100 = np.exp(np.interp(np.log(100), np.log(knots), d["logK_imaging_poisson"]))
g = np.geomspace(50, 1500, 50)
ax.plot(g, K100 * (g / 100) ** -2.3, "--", color=C_REF, lw=1.5)
ax.text(600, K100 * (600 / 100) ** -2.3 * 3, "$\\propto d^{-2.3}$", color="#52514e", fontsize=TICK_FS + 1)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(f"Receiver\u2013sender distance ({MU})", fontsize=LABEL_FS)
ax.set_ylabel("Transfer kernel $K(d)$", fontsize=LABEL_FS)
ax.legend(fontsize=TICK_FS, frameon=False, loc="lower left")
style(ax); letter(ax, "A", dx=-0.22)

ax = axes[1]
ax.plot(d["umi_cdf_d"], d["umi_cdf"], color=C_METHOD, lw=2)
f30 = np.interp(30, d["umi_cdf_d"], d["umi_cdf"])
ax.plot([30, 30], [0, f30], ":", color=C_REF); ax.plot([1, 30], [f30, f30], ":", color=C_REF)
ax.text(34, f30 - 0.08, f"{100 * f30:.0f}% of UMIs from\nsenders within 30 {MU}", fontsize=TICK_FS)
ax.set_xscale("log"); ax.set_xlim(1, 3000); ax.set_ylim(0, 1)
ax.set_xlabel(f"Distance to sender ({MU}, imaging)", fontsize=LABEL_FS)
ax.set_ylabel("Cumulative fraction of UMIs", fontsize=LABEL_FS)
style(ax); letter(ax, "B", dx=-0.22)

fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "FigS2_final.png"), dpi=300, bbox_inches="tight")
plt.show()

### Supp Fig S3: the counts identify local contacts but barely constrain regional geometry

(A) **Local information is strong.** Each held-out receiver localized by maximum likelihood with the true sender positions: 5.7 µm median error with the Poisson likelihood that accumap uses, worse with likelihoods that discount large counts. (B) **UMAP's error is regional.** Displacements of beads hundreds of micrometers apart are strongly correlated: UMAP's error is mostly a smooth warp. (C) **The counts barely detect that warp.** Removing UMAP's smooth error cuts the median error from 38.6 to about 10-13 µm but recovers only about 1% of the likelihood gap to the truth. Hence the priors: uniform density detects compression and stretching; isotropic contact directions detect shear.

In [ ]:
# ================ Supp Fig S3: what the counts identify (local vs regional) ==
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
ax = axes[0]
for key, col, lab in (("real_poisson", C_METHOD, "Poisson"), ("real_nb", C_ALT, "negative binomial"),
                      ("real_bern", C_ALT2, "detection only"), ("umap", C_UMAP, "UMAP")):
    e = d[f"oracle_err_{key}"]
    cdf(ax, e, col)
    m = np.median(e)
    ax.lines[-1].set_label(f"{lab} (median {m:.1f} {MU})")
    ax.plot([m], [0.5], "o", color=col, ms=6, mec="white", mew=1)
ax.legend(fontsize=TICK_FS, frameon=False, loc="lower right")
ax.set_xlim(0, 100); ax.set_ylim(0, 1)
ax.set_xlabel(f"Held-out receiver error ({MU})", fontsize=LABEL_FS)
ax.set_ylabel("Fraction of receivers", fontsize=LABEL_FS)
style(ax); letter(ax, "A", dx=-0.2)

ax = axes[1]
mids = [np.sqrt(lo * hi) for lo, hi in COV_BINS]
ax.axhline(0, color=C_REF, lw=0.8)
ax.plot(mids, COV_UMAP, "o-", color=C_UMAP, lw=2, ms=7, mec="white", mew=1.5)
ax.plot(mids, COV_IID, "s-", color=C_REF, lw=2, ms=7, mec="white", mew=1.5)
ax.text(mids[1] * 1.15, COV_UMAP[1] + 0.03, "UMAP", color=C_UMAP, fontsize=TICK_FS + 1, fontweight="bold")
ax.text(mids[0] * 0.95, 0.07, f"independent errors ($\\sigma$ = 30 {MU})", color="#52514e", fontsize=TICK_FS)
ax.set_xscale("log")
ax.set_xlabel(f"True separation of the two beads ({MU})", fontsize=LABEL_FS)
ax.set_ylabel("Displacement covariance\n(normalized by mean squared error)", fontsize=LABEL_FS)
style(ax); letter(ax, "B", dx=-0.2)

ax = axes[2]
for k, (lab, gain, med) in enumerate(DECOMP):
    col = C_UMAP if k == 0 else (C_METHOD if lab == "truth" else C_ALT)
    ax.plot([gain], [med], "o", color=col, ms=9, mec="white", mew=1.5)
ax.text(4, 38.6, "UMAP", fontsize=TICK_FS, va="center")
ax.text(96, 2.5, "truth", fontsize=TICK_FS, ha="right")
ax.annotate("UMAP + the true smooth warp\n(control grids 400, 200, 100, 50 " + MU + ")",
            xy=(2, 11.5), xytext=(14, 20), fontsize=TICK_FS,
            arrowprops=dict(arrowstyle="-", color=C_REF, lw=0.8))
ax.set_xlim(-10, 110); ax.set_ylim(-3, 45)
ax.set_xlabel("Log-likelihood gain recovered (% of UMAP $\\to$ truth)", fontsize=LABEL_FS)
ax.set_ylabel(f"Median per-bead error ({MU})", fontsize=LABEL_FS)
style(ax); letter(ax, "C", dx=-0.2)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "FigS3_final.png"), dpi=300, bbox_inches="tight")
plt.show()

### Supp Table S3: likelihood alone cannot escape UMAP's error

Before adding physical priors, we tested whether the count likelihood alone could correct UMAP's layout. Receiver and sender positions, sender abundances and receiver capture were optimized jointly under the Poisson count model on c58_8, with the kernel and background fixed at their imaging fit (a better kernel than is available without imaging), by several search strategies. Started at the imaging positions, the optimization stays near them, and independent per-bead errors of UMAP's median size are largely corrected; but no strategy escapes UMAP's own error, because that error is a smooth regional warp to which the counts are nearly blind (Supp Fig S3).

The searches were development code (`joint_proto.py`, `patch_moves.py` and `warp_opt.py` in the paper's development scripts) and are not rerun here; their final layouts are cached and scored below.

In [ ]:
J = np.load(os.path.join(RESULTS, "joint_search", "c58_8_joint_search.npz"))
T = d["c58_8_truth"]
med = lambda k: evaluate(J[k], T)["median"]
rows = [("UMAP embedding", "none", "umap_start"),
        ("UMAP embedding", "block coordinate ascent, 16 rounds", "bca"),
        ("UMAP embedding", "kernel annealing (Gaussian blur, 60 -> 0 um), maximum-likelihood updates", "anneal_argmax"),
        ("UMAP embedding", "kernel annealing, posterior-mean updates", "anneal_mean"),
        ("UMAP embedding", "posterior-mean updates, no annealing", "mean_noanneal"),
        ("UMAP embedding", "rigid patch translations (patches 480 -> 40 um)", "patch_moves"),
        ("UMAP embedding", "smooth displacement field, gradient ascent (grids 400 -> 50 um)", "warp"),
        (f"imaging + independent noise (sigma = 30 um; {med('noise30_start'):.1f})", "block coordinate ascent, 12 rounds", "noise30"),
        ("imaging positions", "block coordinate ascent, 12 rounds", "from_truth")]
pd.DataFrame([{"start": s, "search": m, "median error (um)": round(med(k), 1)} for s, m, k in rows]).set_index(["start", "search"])